In [1]:
import numpy as np
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm
import random




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, test=False):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.test = test

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.annotations.iloc[idx, 0] + ".png")
        # Keep identical core logic; just ensure robust RGB conversion (already present)
        image = Image.open(img_name).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.test:
            return image
        else:
            label = int(self.annotations.iloc[idx, 1])
            return image, label




In [3]:
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)



In [4]:
test_csv_file = "/kaggle/input/aptos2019-blindness-detection/test.csv"
test_root_dir = "/kaggle/input/aptos2019-blindness-detection/test_images"
test_dataset = BlindnessDataset(
    test_csv_file, test_root_dir, transform=transform, test=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)



In [5]:
"""
model_paths = {
    'resnet18': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/resnet18(WD_1e-3)_aptos.pth",
    #'efficientnet_b5': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/efficientnet_b5.pth",
    #'inception_resnet_v2': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/inception_resnet_v2.pth",
    #'inception_v4': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/inception_v4.pth",
    'seresnext50_32x4d': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext50_32x4d.pth",
    'seresnext101_32x4d': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext101_32x4d.pth"
}
"""
model_paths = {
    "seresnext50_32x4d": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext50_32x4d.pth",
}

model_names = {
    "resnet18": "resnet18",
    "efficientnet_b5": "efficientnet_b5",
    "inception_resnet_v2": "inception_resnet_v2",
    "inception_v4": "inception_v4",
    "seresnext50_32x4d": "seresnext50_32x4d",
    "seresnext101_32x4d": "seresnext101_32x4d",
}



In [6]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models_list = []

for model_key, path in model_paths.items():
    model_name = model_names[model_key]
    if os.path.exists(path):
        model = timm.create_model(model_name, pretrained=False, num_classes=5)
        state = torch.load(path, map_location="cpu")
        model.load_state_dict(state)
        model.to(device)
        model.eval()
        models_list.append(model)
    else:
        print(
            f"Warning: weights not found at {path}. Will use fallback training if no models load."
        )

fallback_model_name = model_names.get("seresnext50_32x4d", "seresnext50_32x4d")
fallback_model = None

if len(models_list) == 0:
    print("No pretrained models loaded. Starting fallback training on train.csv ...")
    train_csv_file = "/kaggle/input/aptos2019-blindness-detection/train.csv"
    train_root_dir = "/kaggle/input/aptos2019-blindness-detection/train_images"
    train_dataset = BlindnessDataset(
        train_csv_file, train_root_dir, transform=transform, test=False
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=16,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    fallback_model = timm.create_model(
        fallback_model_name, pretrained=True, num_classes=5
    )
    fallback_model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        fallback_model.parameters(), lr=1e-4, weight_decay=1e-4
    )

    fallback_model.train()
    epochs = 1
    for epoch in range(epochs):
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Train epoch {epoch+1}/{epochs}")
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = fallback_model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            pbar.set_postfix(loss=running_loss / max(1, (pbar.n + 1)))
    fallback_model.eval()
    models_list = [fallback_model]

print(f"Models ready: {len(models_list)}")



No pretrained models loaded. Starting fallback training on train.csv ...


model.safetensors:   0%|          | 0.00/111M [00:00<?, ?B/s]

Train epoch 1/1: 100%|██████████| 206/206 [02:17<00:00,  1.50it/s, loss=0.758]

Models ready: 1


In [7]:
all_outputs = []

with torch.no_grad():
    for images in tqdm(test_loader, desc="Inference"):
        images = images.to(device, non_blocking=True)
        outputs = [model(images).unsqueeze(0) for model in models_list]
        outputs = torch.cat(outputs, dim=0)
        averaged_outputs = torch.mean(outputs, dim=0)
        all_outputs.append(averaged_outputs.cpu().numpy())

all_outputs = np.concatenate(all_outputs, axis=0)

probs = torch.softmax(torch.from_numpy(all_outputs), dim=1).numpy()
class_ids = np.arange(5, dtype=np.float32)[None, :]  # shape (1,5)
expected_score = (probs * class_ids).sum(axis=1)  # shape (N,)

raw_pred = np.rint(expected_score).astype(np.int64)
raw_pred = np.clip(raw_pred, 0, 4)

# Change: use a single mapping table to avoid sequential remap side-effects that can
# accidentally collapse everything into one class and/or create invalid outputs -> fixes 0.0 (invalid) submissions.
# Keep the same intended "pull-to-middle" behavior: 0->1, 1->2, 2->2, 3->2, 4->3
lut = np.array([1, 2, 2, 2, 3], dtype=np.int64)
final_predictions = lut[raw_pred]

# Change: final defensive clipping + dtype enforcement to guarantee a valid Kaggle submission column.
final_predictions = np.clip(final_predictions, 0, 4).astype(np.int64)

print(
    "Predictions:",
    final_predictions.shape,
    int(final_predictions.min()),
    int(final_predictions.max()),
    "unique:",
    np.unique(final_predictions),
)



Inference: 100%|██████████| 23/23 [00:17<00:00,  1.34it/s]

Predictions: (367,) 1 2 unique: [1 2]


In [8]:
test_ids = pd.read_csv(test_csv_file)["id_code"].values

# Change: assert alignment/length to prevent writing an invalid submission that scores 0.0.
assert len(final_predictions) == len(
    test_ids
), f"Prediction length mismatch: preds={len(final_predictions)} vs test={len(test_ids)}"

submission_df = pd.DataFrame(
    {
        "id_code": test_ids,
        "diagnosis": final_predictions.astype(np.int64),
    }
)

# Extra safety: ensure exact column order and integer dtype
submission_df = submission_df[["id_code", "diagnosis"]]
submission_df["diagnosis"] = submission_df["diagnosis"].astype(int)

submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print("Wrote:", submission_path, "rows:", len(submission_df))
print(submission_df.head())

Wrote: submission.csv rows: 367
        id_code  diagnosis
0  b460ca9fa26f          1
1  6cee2e148520          1
2  ca6842bfcbc9          2
3  6cbc3dad809c          2
4  a9bc2f892cb3          1
